In [1]:
# Check GPU availability and library versions

import torch
import transformers

print("=" * 50)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("No GPU detected!")

print("=" * 50)

Torch: 2.10.0+cu128
Transformers: 4.46.3
CUDA Available: True
GPU: Tesla T4
CUDA: 12.8


In [2]:
# Clone CyberGuard-LLM repository

!git clone https://github.com/ifra817/CyberGuard-LLM.git

fatal: destination path 'CyberGuard-LLM' already exists and is not an empty directory.


In [1]:
# Cell 3 — Install Dependencies
!pip install -q -U \
transformers==4.46.3 \
trl==0.12.2 \
peft==0.13.2 \
accelerate==1.1.1 \
bitsandbytes \
triton \
datasets==3.1.0 \
sentencepiece

In [ ]:
# Restart notebook so newly installed packages are loaded

import os
os._exit(0)

In [2]:
# Import all libraries used for fine-tuning

import json
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from trl import SFTTrainer

print("Everything imported successfully.")

Everything imported successfully.


In [3]:
# Verify installed package versions

import transformers
import accelerate
import peft
import trl
import bitsandbytes

print("=" * 60)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("=" * 60)

Torch: 2.10.0+cu128
Transformers: 4.46.3
TRL: 0.12.2
PEFT: 0.13.2
Accelerate: 1.1.1
BitsAndBytes: 0.50.0


In [18]:
# Login to Hugging Face to access gated Llama models

from huggingface_hub import login

login()

In [5]:
# Generate processed training dataset

%cd CyberGuard-LLM

!python training/dataset/dataset.py

/kaggle/working/CyberGuard-LLM
Loading raw dataset from hf://datasets/AlicanKiraz0/Cybersecurity-Dataset-Heimdall-v1.1/train-set-conversations.json...
Loaded 21257 total conversation instances.
Validating messages...
Validation complete. Retained 21256 valid conversations.
Formatting conversations into Llama-3 prompt template...
Saving processed data to /kaggle/working/CyberGuard-LLM/training/dataset/processed/heimdall_llama3_processed.json...
Dataset processing complete!


In [6]:
# Load processed dataset

from datasets import Dataset
import json

with open(
    "training/dataset/processed/heimdall_llama3_processed.json",
    "r",
    encoding="utf-8",
) as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

dataset = dataset.remove_columns(["id", "messages"])

print(dataset)
print()
print(dataset[0]["text"][:400])

Dataset({
    features: ['text'],
    num_rows: 21256
})

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a highly specialized AI assistant for advanced cyber-defense whose mission is to deliver accurate, in-depth, actionable guidance on information-security principles—confidentiality, integrity, availability, authenticity, non-repudiation, and privacy—by offering concise executive summaries that drill down into technical detail, ind


In [7]:
# Load Llama 3.2 3B model using 4-bit quantization

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.


In [8]:
# Attach LoRA adapters

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [9]:
# Verify model and LoRA configuration

print("Trainable parameters:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        print(param.dtype)
        break

print()

print("Dataset size:", len(dataset))
print("Device:", model.device)

Trainable parameters:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
torch.float32

Dataset size: 21256
Device: cuda:0


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    
    # Cap total training steps so it finishes quickly!
    max_steps=500,  
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    
    # Save a checkpoint every 100 steps just in case!
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,
    report_to="none",
)

In [11]:
# Fine-tune the model using SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/21256 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can

Step,Training Loss
10,2.809100
20,2.408000
30,1.624500
40,1.320400
50,1.262300
60,1.248600
70,1.229700
80,1.141200
90,1.135300
100,1.100800


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=500, training_loss=1.0916959857940673, metrics={'train_runtime': 2922.2554, 'train_samples_per_second': 0.684, 'train_steps_per_second': 0.171, 'total_flos': 2.324313449628672e+16, 'train_loss': 1.0916959857940673, 'epoch': 0.0940910801656003})

In [12]:
# Save LoRA adapter and tokenizer

trainer.model.save_pretrained("./cyberguard-lora")
tokenizer.save_pretrained("./cyberguard-lora")

print("LoRA adapter saved.")

LoRA adapter saved.


In [13]:
# Merge LoRA weights into the base model

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

merged = PeftModel.from_pretrained(
    base_model,
    "./cyberguard-lora",
)

merged = merged.merge_and_unload()

merged.save_pretrained("./CyberGuard-LLM")
tokenizer.save_pretrained("./CyberGuard-LLM")

print("Merged model saved.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model saved.


In [14]:
# Run a simple inference test

prompt = "Explain SQL Injection."

inputs = tokenizer(prompt, return_tensors="pt").to(merged.device)

outputs = merged.generate(
    **inputs,
    max_new_tokens=200,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Explain SQL Injection. What are the possible ways of injecting code into a database?
SQL Injection occurs when an attacker injects malicious SQL code into a web application's database. This allows them to manipulate data and potentially gain unauthorized access to sensitive information.
There are several ways an attacker can inject SQL code into a web application's database, including:
1. User Input: An attacker can inject SQL code by entering malicious input into a form or field on a web page. For example, if a web application allows users to input search queries, an attacker can enter a query that includes malicious SQL code.
2. Parameterized Queries: If a web application uses parameterized queries to interact with the database, an attacker can still inject SQL code by manipulating the parameters passed to the query.
3. Dynamic SQL: If a web application generates dynamic SQL code based on user input, an attacker can inject malicious SQL code into the generated query.
4. Error Handlin

In [19]:
# Push adapter & tokenizer to your brand new HF repo
trainer.model.push_to_hub("ifra817/CyberGuard-FinetunedLlama")
tokenizer.push_to_hub("ifra817/CyberGuard-FinetunedLlama")

print("CyberGuard-FinetunedLlama uploaded successfully!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CyberGuard-FinetunedLlama uploaded successfully!


In [20]:
!zip -r CyberGuard-FinetunedLlama.zip ./cyberguard-lora

updating: cyberguard-lora/ (stored 0%)
updating: cyberguard-lora/adapter_config.json (deflated 53%)
updating: cyberguard-lora/README.md (deflated 66%)
updating: cyberguard-lora/special_tokens_map.json (deflated 63%)
updating: cyberguard-lora/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
updating: cyberguard-lora/tokenizer_config.json (deflated 94%)
updating: cyberguard-lora/tokenizer.json (deflated 85%)
